# Analysis of U.S. International Trade Balance

Fall 2024 Data Science Project

Akshith Kantareddy

# Introduction

The United States is a global leader in international trade, engaging in the exchange of goods and services with countries worldwide. In these economically tubulent times, understanding the dynamics of U.S. trade is very important for economic policy decisions, and international relations. This project analyzes U.S. international trade data, focusing on the balance of trade in goods and services. It aims to explore the trends and relationships between imports and exports and analyze the factors that influence trade balance. More specifically we want to see how has overall trade balance changed over time, what are the factors contributing to this trend, is there a significant relationship between the values of American imports and exports for different trading partners, how do imports and exports in the service sector compare to the goods sector, and what are the implications for the overall trade balance? The answers to these questions can be used to affect economic policy by policymakers, but also can be analyzed by individuals to better understand aspects of life like the stock market, grocery store pricing, the availability of jobs in certain sectors, etc. By analyzing the historical data and using statistical methods this project aims to shed light on the dynamics of international trade. For this we will be using International Trades and Goods Dataset from the U.S. Bureau of Economic Analysis (https://www.bea.gov/data/intl-trade-investment/international-trade-goods-and-services).

Here are the libraries we need to import:

In [ ]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.neighbors import NearestNeighbors

# Data Curation

First we need to import some tables from the dataset as dataframes. This can be done with pd.read_excel. For now we will have one dataframe for service imports and another for exports. Note that because the dataset orginally cannot be directly read, as the columns will be unamned, we can create new sheets in the dataset and manually and copy and paste the columns and the datapoints for correct formatting.

In [ ]:
export_df = pd.read_excel("trad-time-series-0824 - Copy.xlsx", sheet_name="Services")
import_df = pd.read_excel("trad-time-series-0824 - Copy.xlsx", sheet_name="Ser_Imports")

balance_df = pd.read_excel("trad-time-series-0824 - Copy.xlsx", sheet_name="Balance")
gs_export_df = pd.read_excel("trad-time-series-0824 - Copy.xlsx", sheet_name="Exports")
gs_import_df = pd.read_excel("trad-time-series-0824 - Copy.xlsx", sheet_name="Imports")

countries_2024_df = pd.read_excel("trad0824.xlsx", sheet_name="countries-2024")
countries_2023_df = pd.read_excel("trad0824.xlsx", sheet_name="countries-2023")

FileNotFoundError: [Errno 2] No such file or directory: 'trad-time-series-0824 - Copy.xlsx'

**Data Cleaning**

Before we do data analysis, we need to clean the datasets to prepare them for use. First we need to change the serivces export_df as there are some values where we cannot access the total due to null values in columns that are not Passenger Fares or Travel (The 2 most significant contributors to total). So we will replace the null total values with the sums of Passenger Fares and Travel.

In [ ]:
pd.set_option('future.no_silent_downcasting', True)
export_df['Total'] = export_df['Total'].replace('(1)', 1)

def calculate_total(row):
    if row['Total'] == 1:
        passenger_fares = pd.to_numeric(row['Passenger Fares'], errors='coerce')
        travel = pd.to_numeric(row['Travel'], errors='coerce')
        total = passenger_fares + travel
        return total
    else:
        return row['Total']

export_df['Total'] = export_df.apply(calculate_total, axis=1)
export_df['Total'] = pd.to_numeric(export_df['Total'], errors='coerce')

Also we have to make sure the dates in the Period columns are date objects for later use.

In [ ]:
index_a = balance_df[balance_df['Period'] == "2024 Jul (R)"].index[0]
balance_df.loc[index_a, 'Period'] = "2024 Jul"

index_b = gs_export_df[gs_export_df['Period'] == "2024 Jul (R)"].index[0]
gs_export_df.loc[index_b, 'Period'] = "2024 Jul"

index_c = gs_import_df[gs_import_df['Period'] == "2024 Jul (R)"].index[0]
gs_import_df.loc[index_c, 'Period'] = "2024 Jul"

def convert_dates(df):
    month_numbers = []
    current_year = None
    month_count = 0

    for period_str in df['Period']:
        year, month_abbr = period_str.split()
        year = int(year)

        if year != current_year:
            current_year = year
            month_count = 1
        else:
            month_count += 1

        date_str = f"{year}-{month_count:02}"
        month_numbers.append(pd.to_datetime(date_str, format='%Y-%m'))

    df['Period'] = month_numbers
    return df

balance_df = convert_dates(balance_df)
gs_export_df = convert_dates(gs_export_df)
gs_import_df = convert_dates(gs_import_df)

Now we are ready for data analysis.

# Exploratory Data Analysis

Let's take a look at some data to see if we can see anything directly.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(balance_df['Period'], balance_df['Total'], label='Trade Balance')
plt.title('Trade Balance Over Time')
plt.xlabel('Period')
plt.ylabel('Balance')
plt.grid(True)
plt.legend()
plt.show()

We can see that generally the trade balance is pretty volatile and dips during economically down years like 2008 or 2020. We can do hypothesis testing to take a closer look and find relationships.

First let's look at the service import and export datasets (export_df and import_df) and perform some hypothesis test to get a better understanding of the realtionships between the data which we can use later. We will first perform a Two Sample T-Test to observe the relationship between travel exports and imports.

**Two Sample T-Test**

H0: The trade of travels is the same between imports and exports

HA: The trade of travels is different between imports and exports

In [ ]:
t_statistic, p_value = stats.ttest_ind(import_df['Travel 1'], export_df['Travel'])
print(p_value)

Since the p-value is greater than the significance level of 0.05, the null hypothosis fails to be rejected.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=['Imports'] * len(import_df['Travel 1']) + ['Exports'] * len(export_df['Travel']),y=import_df['Travel 1'].tolist() + export_df['Travel'].tolist())
plt.title('Distribution of Travel for Imports and Exports')
plt.ylabel('Travel')
plt.show()

Let's try doing the same thing using the data from the total column instead to compare all services

In [ ]:
t_statistic, p_value = stats.ttest_ind(import_df['Total'], export_df['Total'])
print(p_value)

plt.figure(figsize=(8, 6))
sns.boxplot(x=['Imports'] * len(import_df['Total']) + ['Exports'] * len(export_df['Total']),y=import_df['Total'].tolist() + export_df['Total'].tolist())
plt.title('Distribution of Total Services for Imports and Exports')
plt.ylabel('Total')
plt.show()

We can see from the two tests that while the means of travel exports and imports are somewhat similar, for overall services they differ a lot.

**One Tailed Test**
According to the U.S. Travel Association (https://www.ustravel.org/system/files/media_root/document/Research_Fact-Sheet_US-Travel-Answer-Sheet.pdf) the monthly average of U.S. Travel exports were approximatley $12 billion (12,000 million).

H0: The mean of export travel is equal to $12,000 million

HA: The mean of export travel is not equal to $12,000 million.

In [ ]:
t_statistic, p_value = stats.ttest_1samp(export_df['Travel'], popmean=12000)
print(p_value)

Since the p-value is less then 0.05, the null hypothesis is rejected

In [ ]:
sns.histplot(export_df['Travel'], kde=True)
plt.title('Distribution of Export Travel')
plt.xlabel('Travel Exports (Millions of Dollars)')
plt.ylabel('Frequency')
plt.axvline(x=12000, color='red', linestyle='--', label='Population Mean (12000)')
plt.legend()
plt.show()

Let's do a Two-Sample T-Test for Goods and Services Trade Balance, since we have been only focused on services so far.

H0: The mean trade balance for goods is equal to the mean trade balance for services.

HA: The mean trade balance for goods is not equal to the mean trade balance for services.

In [ ]:
t_statistic, p_value = stats.ttest_ind(balance_df['Goods'], balance_df['Services'])
print(p_value)

plt.figure(figsize=(8, 6))
sns.boxplot(data=balance_df[['Goods', 'Services']])
plt.xticks([0, 1], ['Goods Trade Balance', 'Services Trade Balance'])
plt.title('Distribution of Goods and Services Trade Balances')
plt.ylabel('Balance (Millions of Dollars)')
plt.show()

Here we can see something very interesting where the U.S. generally exports more services while importing goods.

Now let's test to see the changes in travel exports and imports between 2023 and 2024

Two-Sample T-Test for Total Exports

H0: The mean total exports in 2024 are equal to the mean total exports in 2023.

HA: The mean total exports in 2024 are not equal to the mean total exports in 2023.

In [ ]:
t_statistic, p_value = stats.ttest_ind(countries_2024_df['Exports'], countries_2023_df['Exports'])
print("P-value for two-sample t-test:", p_value)

plt.figure(figsize=(8, 6))
sns.boxplot(x=['2024'] * len(countries_2024_df['Exports'])+ ['2023'] * len(countries_2023_df['Exports']),
    y=countries_2024_df['Exports'].tolist()+ countries_2023_df['Exports'].tolist(),)
plt.title('Distribution of Total Exports for 2024 and 2023')
plt.ylabel('Total Exports')
plt.show()

Now lets do the same for imports

In [ ]:
t_statistic, p_value = stats.ttest_ind(
    countries_2024_df['Imports'], countries_2023_df['Imports']
)
print("P-value for two-sample t-test:", p_value)

plt.figure(figsize=(8, 6))
sns.boxplot(x=['2024'] * len(countries_2024_df['Imports'])+ ['2023'] * len(countries_2023_df['Imports']),
    y=countries_2024_df['Imports'].tolist()+ countries_2023_df['Imports'].tolist(),)
plt.title('Distribution of Total Imports for 2024 and 2023')
plt.ylabel('Total Imports')
plt.show()

# Primary analysis

For primary analysis, we will be utilizing linear regression because it will allow us to better see the relationships between certain data and allow us to make predictions about the future. Along with this we will be using K-Fold validation to account for overfitting and make sure the linear regression model is perfroming well. We will first do this on the general trade balance data.

In [ ]:
X = balance_df[['Period']].copy()
X['Period'] = X['Period'].apply(lambda date: date.toordinal())
y = balance_df['Total']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

print(f"R-squared: {np.mean(r2_scores)}")

One aspect we can observe from this is that the r^2 value is closer to 1 than 0, so the linear model explains a larger amount of variation in the balance data.  

Next we will do the same to see if the amount of countries where exports are similar to imports.

In [ ]:
countries_df = pd.concat([countries_2023_df, countries_2024_df])

X = countries_df[['Exports']]
y = countries_df['Imports']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

print(f"R-squared: {np.mean(r2_scores)}")

Since r^2 is close to 1, we can see the model will explain variation in the data.

# Visualization

Let's visualize the results of the primary analysis.

In [ ]:
X = balance_df[['Period']].copy()
X['Period'] = X['Period'].apply(lambda date: date.toordinal())
y = balance_df['Total']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
X_test_dates = X_test['Period'].apply(lambda ordinal: datetime.fromordinal(ordinal))
plt.scatter(X_test_dates, y_test, color='blue', label='Actual Data')
plt.plot(X_test_dates, y_pred, color='red', linewidth=2, label='Regression Line')
plt.title('Linear Regression Model for Trade Balance')
plt.xlabel('Period')
plt.ylabel('Trade Balance')
plt.legend()
plt.grid(True)
plt.show()

k = 5
nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree').fit(X)
distances, indices = nbrs.kneighbors(X)

plt.figure(figsize=(10, 8))
plt.scatter(X['Period'], y, color='blue', label='Data Points')

for i in range(X.shape[0]):
    for j in indices[i, 1:]:
        plt.plot(
            [X['Period'].iloc[i], X['Period'].iloc[j]],
            [y.iloc[i], y.iloc[j]],
            color='gray',
            linewidth=0.5,
        )

plt.title('knn Graph')
plt.xlabel('Period (Ordinal)')
plt.ylabel('Balance')
plt.legend()
plt.show()

The slope of the line is negative indicating a general trend of overall trade balance decreasing for the United States. Continually the large amount of clustering indicates that the mid-2010's had similar balances.

In [ ]:
countries_df = pd.concat([countries_2023_df, countries_2024_df])

X = countries_df[['Exports']]
y = countries_df['Imports']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
plt.scatter(X_test, y_test, color='blue', label='Actual Data')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Regression Line')
plt.title('Exports vs. Imports')
plt.xlabel('Exports')
plt.ylabel('Imports')
plt.legend()
plt.grid(True)
plt.show()

From this we can see that when trading with a country, the U.S. generally imports more than it exports even when exports increase.

# Conclusion

After the multiple forms of data analysis conducted, we can see that:

1) The overall trade balance of the United States has generally been decreasing over time, as indicated by the negative slope of the linear regression model. This suggests that the U.S. is importing more than it is exporting, leading to a trade deficit.

2) here is a strong positive correlation between the values of American imports and exports for different trading partners. This means that as exports to a country increase, imports from that country also tend to increase. However, the linear regression model shows that even when exports increase, the U.S. generally imports more than it exports, indicating a consistent trade deficit with individual trading partners.

3)  The U.S. generally exports more services than it imports, resulting in a trade surplus in the service sector. Conversely, the U.S. imports significantly more goods than it exports, leading to a large trade deficit in the goods sector. This suggests that the United States is a service based economy